# Inside the Tokenizer

Starter notebook. Fill in the TODOs and **run every cell** before committing — the graders need to see your outputs.


## Task 1 — Tokenize and compare

Pick a paragraph of mixed content (English + code + another language + an emoji), tokenize with **two** tokenizers, and show tokens, IDs, and counts side by side.


In [1]:
import tiktoken

TEXT = """Hello! I am learning how LLM tokenizers work.
Here is a Python line: print("Tokenization is weird!")
Azerbaijani sentence: Mən bu labı addım-addım yazıram.
Emoji test: 🚀🔥😊
"""

# Two different BPE tokenizers from tiktoken
enc_cl100k = tiktoken.get_encoding("cl100k_base")
enc_p50k = tiktoken.get_encoding("p50k_base")

def show_tokenization(name, encoder, text):
    ids = encoder.encode(text)
    tokens = [
        encoder.decode_single_token_bytes(i).decode("utf-8", errors="replace")
        for i in ids
    ]

    print("=" * 80)
    print(name)
    print("Token count:", len(ids))
    print("\nFirst 40 tokens:")
    print(tokens[:40])
    print("\nFirst 40 token IDs:")
    print(ids[:40])
    return len(ids)

count_cl100k = show_tokenization("cl100k_base", enc_cl100k, TEXT)
count_p50k = show_tokenization("p50k_base", enc_p50k, TEXT)

print("\nSide-by-side token count table")
print("-" * 40)
print(f"{'Tokenizer':15} | {'Token count'}")
print("-" * 40)
print(f"{'cl100k_base':15} | {count_cl100k}")
print(f"{'p50k_base':15} | {count_p50k}")

cl100k_base
Token count: 57

First 40 tokens:
['Hello', '!', ' I', ' am', ' learning', ' how', ' L', 'LM', ' token', 'izers', ' work', '.\n', 'Here', ' is', ' a', ' Python', ' line', ':', ' print', '("', 'Token', 'ization', ' is', ' weird', '!")\n', 'A', 'zerbai', 'j', 'ani', ' sentence', ':', ' M', 'ə', 'n', ' bu', ' lab', 'ı', ' add', 'ım', '-add']

First 40 token IDs:
[9906, 0, 358, 1097, 6975, 1268, 445, 11237, 4037, 12509, 990, 627, 8586, 374, 264, 13325, 1584, 25, 1194, 446, 3404, 2065, 374, 16682, 23849, 32, 57860, 73, 5676, 11914, 25, 386, 99638, 77, 1048, 10278, 3862, 923, 38404, 19082]
p50k_base
Token count: 70

First 40 tokens:
['Hello', '!', ' I', ' am', ' learning', ' how', ' LL', 'M', ' token', 'izers', ' work', '.', '\n', 'Here', ' is', ' a', ' Python', ' line', ':', ' print', '("', 'Token', 'ization', ' is', ' weird', '!"', ')', '\n', 'A', 'zer', 'b', 'ai', 'j', 'ani', ' sentence', ':', ' M', '\ufffd', '\ufffd', 'n']

First 40 token IDs:
[15496, 0, 314, 716, 4673, 703, 

**Where do the two tokenizers disagree most, and why?**

The two tokenizers disagree most on the Azerbaijani sentence and some formatting details. 
`cl100k_base` uses 57 tokens, while `p50k_base` uses 70 tokens for the same text, so `cl100k_base` is more efficient here.

The biggest difference appears around the Azerbaijani word parts. 
For example, `cl100k_base` can represent characters like `ə` and word parts such as `zerbai` more cleanly, while `p50k_base` splits the same area into smaller pieces and even shows replacement characters like `�`. 
This means `p50k_base` has more trouble with non-English characters in this text.

They also split punctuation and newlines slightly differently. 
For example, `cl100k_base` keeps some punctuation together like `!")\n`, but `p50k_base` splits it into smaller tokens like `!"`, `)`, and `\n`.

Overall, this shows that tokenizers do not read text like humans do. 
Two tokenizers can see the same sentence very differently, especially when the text contains non-English letters, code-style punctuation, or emojis.


## Task 2 — Hunt the weird cases

Investigate **at least three**: rare vs common word, English vs another language, code vs prose, leading-space, emoji/accents. Show the token counts that prove your point.


In [2]:
def count_tokens(text, encoder=enc_cl100k):
    return len(encoder.encode(text))

cases = {
    "common_word": "internationalization",
    "rare_made_up_word": "flibbertigibbetizationxyz",

    "english_sentence": "I am writing this lab step by step.",
    "azerbaijani_sentence": "Mən bu labı addım-addım yazıram.",

    "prose": "The user logs in and sees a dashboard with recent activity.",
    "code": 'if user.is_active:\n    print("Welcome back!")',

    "no_leading_space": "hello",
    "with_leading_space": " hello",

    "normal_text": "I am happy today.",
    "emoji_text": "I am happy today 😊🚀🔥",

    "plain_word": "Cafe",
    "accented_word": "Café",
}

print(f"{'Case':25} | {'Tokens':6} | Text")
print("-" * 80)

for name, text in cases.items():
    print(f"{name:25} | {count_tokens(text):6} | {text!r}")

Case                      | Tokens | Text
--------------------------------------------------------------------------------
common_word               |      2 | 'internationalization'
rare_made_up_word         |      8 | 'flibbertigibbetizationxyz'
english_sentence          |      9 | 'I am writing this lab step by step.'
azerbaijani_sentence      |     14 | 'Mən bu labı addım-addım yazıram.'
prose                     |     12 | 'The user logs in and sees a dashboard with recent activity.'
code                      |     11 | 'if user.is_active:\n    print("Welcome back!")'
no_leading_space          |      1 | 'hello'
with_leading_space        |      1 | ' hello'
normal_text               |      5 | 'I am happy today.'
emoji_text                |     12 | 'I am happy today 😊🚀🔥'
plain_word                |      2 | 'Cafe'
accented_word             |      3 | 'Café'


**Observations (one sentence per case):**

The tokenizer handles common English words more efficiently than rare or made-up words. 
For example, `internationalization` uses only 2 tokens, but `flibbertigibbetizationxyz` uses 8 tokens. 
This shows that common words or common word parts are easier for the tokenizer to recognize.

The Azerbaijani sentence uses more tokens than the English sentence. 
The English sentence uses 9 tokens, while the Azerbaijani sentence uses 14 tokens. 
This happens because non-English letters like `ə`, `ı`, and Azerbaijani word parts are split into smaller pieces.

Emojis also increase the token count a lot. 
The normal sentence `I am happy today.` uses 5 tokens, but the same sentence with emojis uses 12 tokens. 
So emojis may look small to humans, but they can be expensive in token count and eat into the context window quickly.

The accented word `Café` uses 3 tokens, while `Cafe` uses 2 tokens. 
This shows that even one special character can change tokenization, which matters when pricing API calls at scale.

One surprising result is that `hello` and ` hello` both use 1 token in `cl100k_base`. 
This tokenizer encodes the leading space as part of the token itself (a BPE convention), so the space does not add a separate token — meaning it has no cost impact here.

Overall, the weirdest cases are rare words, non-English text, accented characters, and emojis. 
These examples show that tokenizers do not count text the same way humans count words or characters.


## Task 3 — Token + cost estimator


In [3]:
def estimate(text, price_in_per_1k, price_out_per_1k, expected_output_tokens):
    """
    Estimate input tokens and projected total cost.

    price_in_per_1k: price for 1000 input tokens
    price_out_per_1k: price for 1000 output tokens
    expected_output_tokens: how many tokens we expect the model to generate
    """
    input_tokens = len(enc_cl100k.encode(text))

    input_cost = (input_tokens / 1000) * price_in_per_1k
    output_cost = (expected_output_tokens / 1000) * price_out_per_1k
    total_cost = input_cost + output_cost

    print("=" * 70)
    print("Input preview:", text[:100].replace("\n", " "), "...")
    print("Input tokens:", input_tokens)
    print("Expected output tokens:", expected_output_tokens)
    print("Projected total cost: $", round(total_cost, 6))

    return input_tokens, total_cost


# Made-up price table
price_in = 0.075
price_out = 0.30

short_question = "How do I reset my password?"

long_document = """
Large language models are trained to predict the next token.
Before the model sees the text, a tokenizer splits the text into token IDs.
Token count matters because API cost and context window limits are based on tokens, not words.
This is why long documents, code, non-English text, and emojis can become more expensive.
"""

code_file = """
def calculate_total(items):
    total = 0
    for item in items:
        total += item["price"] * item["quantity"]
    return total

print(calculate_total([
    {"price": 10, "quantity": 2},
    {"price": 5, "quantity": 4}
]))
"""

results = {
    "short_question": estimate(short_question, price_in, price_out, 50),
    "long_document": estimate(long_document, price_in, price_out, 150),
    "code_file": estimate(code_file, price_in, price_out, 120),
}

print("\nSummary table")
print("-" * 50)
print(f"{'Input':20} | {'Tokens':8} | {'Cost'}")
print("-" * 50)

for name, (tokens, cost) in results.items():
    print(f"{name:20} | {tokens:8} | ${cost:.6f}")

Input preview: How do I reset my password? ...
Input tokens: 7
Expected output tokens: 50
Projected total cost: $ 0.015525
Input preview:  Large language models are trained to predict the next token. Before the model sees the text, a toke ...
Input tokens: 66
Expected output tokens: 150
Projected total cost: $ 0.04995
Input preview:  def calculate_total(items):     total = 0     for item in items:         total += item["price"] * i ...
Input tokens: 66
Expected output tokens: 120
Projected total cost: $ 0.04095

Summary table
--------------------------------------------------
Input                | Tokens   | Cost
--------------------------------------------------
short_question       |        7 | $0.015525
long_document        |       66 | $0.049950
code_file            |       66 | $0.040950


**Which input is most expensive, and is it what you expected?**

The most expensive input is the `long_document`, with a projected cost of `$0.049950`. 
This is because it has 66 input tokens and the highest expected output length, which is 150 tokens.

The `code_file` also has 66 input tokens, the same as the long document, but its final cost is lower: `$0.040950`. 
This happens because the expected output is only 120 tokens instead of 150 tokens.

The `short_question` is the cheapest input. 
It has only 7 input tokens and costs `$0.015525`, because both the prompt and the expected response are short.

This result makes sense to me. 
The cost depends not only on the input tokens, but also on how many output tokens we expect the model to generate. 
So even if two inputs have the same token count, the one with the longer expected output can be more expensive.